# Multimodal Processing Workflow

This notebook contains the complete workflow for multimodal processing using Voyage AI, including:
1. Pipeline implementation
2. Helper functions
3. Usage examples

In [ ]:
import os
from typing import List, Union, Tuple
import urllib.request
from PIL import Image
from io import BytesIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from voyageai import Client

## 1. Pipeline Implementation

First, let's implement our MultimodalPipeline class that will handle the core functionality.

In [ ]:
class MultimodalPipeline:
    def __init__(self, api_key: str = None, model_name: str = "voyage-multimodal-3"):
        """
        Initialize the multimodal pipeline.
        
        Args:
            api_key: Voyage AI API key. If None, will try to get from environment variable.
            model_name: Name of the Voyage AI model to use.
        """
        if api_key is None:
            api_key = os.getenv('VOYAGE_API_KEY')
            if api_key is None:
                raise ValueError("No API key provided and VOYAGE_API_KEY not found in environment")
        
        self.client = Client(api_key=api_key)
        self.model_name = model_name
    
    def embed_documents(self, 
                       documents: List[List[Union[str, Image.Image]]],
                       normalize: bool = True) -> np.ndarray:
        """
        Generate embeddings for a list of documents.
        Each document can contain interleaved text and images.
        """
        result = self.client.multimodal_embed(
            documents=documents,
            model=self.model_name,
            normalize=normalize
        )
        return result.embeddings
    
    def compute_similarity(self, query_embedding: np.ndarray, document_embeddings: np.ndarray) -> np.ndarray:
        """Compute cosine similarity between query and documents."""
        return np.dot(document_embeddings, query_embedding)
    
    def search_documents(self, 
                        query: List[Union[str, Image.Image]], 
                        documents: List[List[Union[str, Image.Image]]],
                        top_k: int = 5) -> List[Tuple[int, float]]:
        """Search documents using a query."""
        query_embedding = self.embed_documents([query])[0]
        doc_embeddings = self.embed_documents(documents)
        similarities = self.compute_similarity(query_embedding, doc_embeddings)
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        return [(idx, similarities[idx]) for idx in top_indices]
    
    def batch_embed_documents(self, 
                            documents: List[List[Union[str, Image.Image]]], 
                            batch_size: int = 32) -> np.ndarray:
        """Embed documents in batches."""
        embeddings = []
        for i in range(0, len(documents), batch_size):
            batch = documents[i:i + batch_size]
            batch_embeddings = self.embed_documents(batch)
            embeddings.append(batch_embeddings)
        return np.vstack(embeddings)

## 2. Helper Functions

These functions help with loading and displaying images.

In [ ]:
def load_image_from_url(url: str) -> Image.Image:
    """Load an image from a URL"""
    response = urllib.request.urlopen(url)
    img_data = response.read()
    return Image.open(BytesIO(img_data))

def display_image(image: Image.Image) -> None:
    """Display an image using matplotlib"""
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

def display_search_results(results: List[Tuple[int, float]], documents: List[List[Union[str, Image.Image]]]) -> None:
    """Display search results with scores"""
    print("\nSearch Results:")
    for idx, score in results:
        print(f"Document: {documents[idx]}")
        print(f"Similarity Score: {score:.4f}\n")

## 3. Initialize Pipeline

Create an instance of our pipeline using the API key from environment variables.

In [ ]:
# Initialize pipeline
pipeline = MultimodalPipeline()
print(f"Pipeline initialized with model: {pipeline.model_name}")

## 4. Example Usage

### 4.1 Basic Text and Image Embedding

In [ ]:
# Example text and image
text = "A beautiful sunset over the ocean"
image_url = "https://images.unsplash.com/photo-1566241440091-ec10de8db2e1"

try:
    # Load and display image
    image = load_image_from_url(image_url)
    display_image(image)
    
    # Create document with both text and image
    document = [[text, image]]
    
    # Generate embedding
    embedding = pipeline.embed_documents(document)
    print(f"\nEmbedding shape: {embedding.shape}")
    
except Exception as e:
    print(f"Error: {e}")

### 4.2 Document Search Example

In [ ]:
# Create a collection of documents
documents = [
    ["A beautiful sunset over the ocean"],
    ["A busy city street at night"],
    ["Mountains covered in snow"],
    ["A tropical beach with palm trees"]
]

# Search query
query = ["sunset by the sea"]

# Search and display results
results = pipeline.search_documents(query, documents, top_k=2)
display_search_results(results, documents)

### 4.3 Batch Processing Example

In [ ]:
# Create a larger collection
large_documents = [[f"Document {i}"] for i in range(50)]

# Process in batches
embeddings = pipeline.batch_embed_documents(large_documents, batch_size=16)
print(f"\nProcessed {len(large_documents)} documents")
print(f"Final embeddings shape: {embeddings.shape}")

## 5. Custom Examples

Add your own examples and experiments below: